# Red Team, Deploy & Optimize (Part 4)

Welcome to the finish line. Over the last three notebooks you built a **Foundry Expert** agent, gave it tools, wrote custom functions, turned on tracing, and ran evaluations. That's a working agent — but it isn't a *production* agent yet.

In this notebook you:

1. **Red-team** your agent with AI-simulated adversarial attacks
2. **Deploy** it to Azure with `azd` and as a hosted agent
3. **Publish** it to Microsoft 365 Copilot / Teams
4. **Monitor** production traffic with traces and continuous evaluations

### Prerequisites

| Requirement | Why |
|---|---|
| Parts 1–3 completed | You need a working `foundry-expert` agent |
| Azure subscription with a Foundry project | Red teaming and deployment run in the cloud |
| `azd` CLI installed | [Install guide](https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd) |
| `az login` completed | Authentication for deployment |


In [ ]:
%pip install azure-ai-projects --pre azure-identity python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

model_name = os.environ["MODEL_NAME"]
print(f"✅ Connected to project — model: {model_name}")

## 🔴 Red Teaming — Stress-Test Before You Ship

You wouldn't deploy a web app without a security scan. The same rule applies to agents.

**Red teaming** uses AI to simulate adversarial users — people actively trying to make your agent produce harmful content, leak system prompts, or behave in unexpected ways. The `azure-ai-projects` SDK has built-in cloud red teaming so you can run these attacks *before* real users encounter your agent.

### Why red-team before deploying?

- **Responsible AI** — Microsoft requires safety testing before production
- **Find blind spots** — Your eval set tests what you *thought of*. Red teaming tests what you *didn't*
- **Compliance** — Many industries require adversarial testing documentation

### What gets tested

| Attack Strategy | What It Does |
|---|---|
| `Baseline` | Straightforward harmful prompts |
| `Jailbreak` | Prompt injection, role-play tricks, instruction override attempts |
| `Base64` / `ROT13` | Encoded prompts that try to bypass content filters |
| `CharacterSpace` | Adds spaces between characters to evade detection |

### Risk categories scored

| Category | Examples |
|---|---|
| **Violence** | Instructions for harm, weapon-making |
| **Sexual** | Explicit content generation |
| **Self-Harm** | Dangerous advice about self-injury |
| **Hate/Unfairness** | Discriminatory or biased outputs |


In [ ]:
from azure.ai.projects.models import (
    RedTeam,
    AzureOpenAIModelConfiguration,
    AttackStrategy,
    RiskCategory,
)

# Point the red team at your model deployment
target_config = AzureOpenAIModelConfiguration(
    model_deployment_name=model_name,
)

# Configure the red team operation
red_team_agent = RedTeam(
    target=target_config,
    display_name="foundry-expert-red-team",
    risk_categories=[
        RiskCategory.VIOLENCE,
        RiskCategory.SEXUAL,
        RiskCategory.SELF_HARM,
        RiskCategory.HATE_UNFAIRNESS,
    ],
    num_turns=3,
    attack_strategies=[
        AttackStrategy.BASELINE,
        AttackStrategy.EASY,
    ],
)

# Launch the scan in the cloud
red_team_response = project_client.red_teams.create(red_team=red_team_agent)
print(f"🔴 Red team operation started: {red_team_response.name}")
print(f"   Status: {red_team_response.status}")

In [ ]:
import time

# Poll until the scan finishes (usually takes a few minutes)
status = red_team_response.status
while status not in ("Completed", "Failed", "Canceled"):
    time.sleep(30)
    result = project_client.red_teams.get(name=red_team_response.name)
    status = result.status
    print(f"   Status: {status}")

print(f"\n🔴 Red team scan finished: {status}")

# Review the results
result = project_client.red_teams.get(name=red_team_response.name)
print(f"   Name:  {result.name}")
print(f"   Risk categories tested: {result.risk_categories}")
print(f"   Attack strategies used: {result.attack_strategies}")
print()
print("📊 View full results in the Foundry portal:")
print("   Go to your project → Evaluate & Red Team → Red teaming results")

## 🚀 Deploy with Azure Developer CLI (`azd`)

Your agent runs great in a notebook. Now let's get it running in the cloud where others can use it.

[Azure Developer CLI (`azd`)](https://learn.microsoft.com/azure/developer/azure-developer-cli/) is the fastest path from code to cloud. It handles infrastructure provisioning, container builds, and configuration in one command.

The workflow looks like this:

```
azd init          →  Scaffold the project structure
azure.yaml        →  Define your service(s)
azd up            →  Provision + build + deploy in one shot
```

### Step 1 — Initialize your project

```bash
# Run this in your project root (not in the notebook)
azd init
```

This creates the `azure.yaml` file and `.azure/` directory that `azd` needs.


In [ ]:
# This is the azure.yaml you'd place in your project root.
# Shown here for reference — create this file in your repo, not in Python.

azure_yaml = """
name: foundry-expert
metadata:
  template: foundry-agent
services:
  agent:
    host: containerapp
    project: .
"""
print(azure_yaml)

In [ ]:
# Deploy with azd — run this in your terminal, not here
# The command provisions infrastructure, builds your container, and deploys.
#
# !azd up --environment foundry-expert
#
# What happens:
#   1. azd provisions a Container App + Container Registry in Azure
#   2. Builds your agent code into a container image
#   3. Deploys the container and wires up environment variables
#   4. Prints the endpoint URL when done

print("Run this in your terminal:")
print()
print("  azd up --environment foundry-expert")
print()
print("After deployment, azd prints your agent's live endpoint URL.")

## 🏠 Deploy as a Hosted Agent

An alternative to `azd` is **Hosted Agents** — Foundry manages the infrastructure for you. You provide a container image and Foundry handles scaling, networking, and health checks.


In [ ]:
from azure.ai.projects.models import (
    ImageBasedHostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)

# Deploy your agent as a hosted agent
# Replace the image URL with your actual ACR image after building
hosted_agent = project_client.agents.create_version(
    agent_name="foundry-expert",
    description="Foundry Expert agent — searches docs, browses code, runs SDK examples",
    definition=ImageBasedHostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="v1")
        ],
        cpu="1",
        memory="2Gi",
        image="<your-registry>.azurecr.io/foundry-expert:latest",
        environment_variables={
            "AZURE_AI_PROJECT_ENDPOINT": os.environ["PROJECT_ENDPOINT"],
            "MODEL_NAME": model_name,
        },
    ),
)

print(f"🏠 Hosted agent created: {hosted_agent.name} v{hosted_agent.version}")

## 💬 Publish to Microsoft 365 Copilot & Teams

This is the "share with others" step from Part 1 — taken to its full conclusion. Your agent can appear in Teams, Microsoft 365 Copilot, and Copilot Studio.

### Publishing workflow

| Step | What You Do |
|------|------------|
| **1. Register** | In the Foundry portal, go to your agent → *Channels* → *Microsoft 365* |
| **2. Configure channel** | Enable the Teams / M365 Copilot channel and set a display name & description |
| **3. Set up auth** | Configure SSO with Microsoft Entra ID so users sign in with their org credentials |
| **4. Submit for approval** | Your IT admin reviews and approves the agent for your organization |
| **5. Test in Teams** | Open Teams, find your agent in the app catalog, and start chatting |

Once approved, anyone in your organization can `@mention` your agent in Teams — no code required on their end.

> **Tip:** Start with a *test tenant* or a small group before rolling out organization-wide.


## 📡 Production Observability

Your agent is live. Real users are talking to it. Now you need to *see* what's happening.

Production observability has three layers:

| Layer | What It Shows | Where to Find It |
|-------|--------------|-------------------|
| **Traces** | Every conversation turn, tool call, and LLM response | Foundry portal → *Operate* → *Traces* |
| **Continuous Evaluations** | Automated quality & safety scoring on production traffic | Foundry portal → *Evaluate* |
| **Alerts** | Notifications when quality drops or safety issues spike | Azure Monitor / App Insights |

Let's set these up.


In [ ]:
# Traces are automatically collected when your agent runs in Foundry.
# You can view them in the portal or query programmatically.

# Get the Application Insights connection string for your project
app_insights_conn = project_client.telemetry.get_application_insights_connection_string()
print(f"📡 App Insights connected: {app_insights_conn[:50]}...")
print()
print("View traces in the Foundry portal:")
print("  → Your project → Operate → Traces")
print()
print("Each trace shows:")
print("  • User input and agent response")
print("  • Tool calls (which tools, what arguments, what they returned)")
print("  • Latency breakdown (LLM time vs. tool time)")
print("  • Token usage per turn")

## 🔄 Continuous Evaluations

Evaluations in Part 3 ran on a static dataset. **Continuous evaluations** run automatically on live production conversations — scoring every response (or a sample) for quality and safety.

This catches problems that your test set never anticipated.


In [ ]:
from azure.ai.projects.models import (
    EvaluationRule,
    ContinuousEvaluationRuleAction,
    EvaluationRuleFilter,
    EvaluationRuleEventType,
)

# First, create an evaluation definition with safety + quality criteria
data_source_config = {"type": "azure_ai_source", "scenario": "responses"}

testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "violence_detection",
        "evaluator_name": "builtin.violence",
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": model_name},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "task_adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {"deployment_name": model_name},
    },
]

eval_object = openai_client.evals.create(
    name="Foundry Expert — Continuous Eval",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"📋 Evaluation created: {eval_object.id}")

# Now create a rule that triggers this eval on every agent response
continuous_eval_rule = project_client.evaluation_rules.create_or_update(
    id="foundry-expert-continuous-eval",
    evaluation_rule=EvaluationRule(
        display_name="Foundry Expert Quality & Safety",
        description="Runs violence, fluency, and task adherence checks on production responses",
        action=ContinuousEvaluationRuleAction(
            eval_id=eval_object.id,
            max_hourly_runs=100,
        ),
        event_type=EvaluationRuleEventType.RESPONSE_COMPLETED,
        filter=EvaluationRuleFilter(agent_name="foundry-expert"),
        enabled=True,
    ),
)

print(f"🔄 Continuous eval rule active: {continuous_eval_rule.display_name}")
print(f"   Evaluates every response from 'foundry-expert'")
print(f"   Max {continuous_eval_rule.action.max_hourly_runs} runs/hour")

## 🎉 You Did It

You've built a Foundry Expert agent from scratch:

| Part | What You Did |
|------|-------------|
| **Part 1** | Created the agent in the portal with Web Search |
| **Part 2** | Added MCP tools and wrote your first code |
| **Part 3** | Built custom tools, enabled tracing, ran evaluations |
| **Part 4** | Red teamed, deployed, and set up production monitoring |

Your agent can search the web, read official docs, browse GitHub source code, execute real SDK code, and manage its own model deployments — all while being monitored for quality and safety.

### Where to go from here

- **Add more tools** from the [Foundry tool catalog](https://ai.azure.com)
- **Build multi-agent workflows** with `WorkflowAgentDefinition`
- **Fine-tune a model** for your specific domain
- **Connect your own data** with Azure AI Search or File Search
- **Publish to Copilot Studio** for no-code customization by business users

Happy forging! 🔥
